In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

## Question 1. MNL Model

Using minimize from scipy for MLE

In [3]:
df = pd.read_csv('project/data.csv') 

features = [
    'prop_starrating',
    'prop_review_score',
    'prop_brand_bool',
    'prop_location_score',
    'prop_accesibility_score',
    'prop_log_historical_price',
    'price_usd',
    'promotion_flag'
]

X_raw = df[features].values
y = df['booking_bool'].values
srch_ids = df['srch_id'].values

intercept_column = np.ones((X_raw.shape[0], 1))
X = np.hstack((intercept_column, X_raw))

#Search Group bins
unique_srch_ids, group_indices = np.unique(srch_ids, return_inverse=True)

In [4]:
# Optimizer
def neg_log_likelihood(beta, X, y, group_indices):
    # Calculate Utility for every row
    V = np.dot(X, beta)
    
    #Weights
    exp_V = np.exp(V)
    
    # Calculate Denominators (sum of weights per search + 1.0 for Walk Away)
    sum_exp_V = np.bincount(group_indices, weights=exp_V)
    D = 1.0 + sum_exp_V
    
    # The Log-Likelihood Math Shortcut
    sum_log_D = np.sum(np.log(D))
    sum_yV = np.sum(y * V)
    
    # Return the negative score for Scipy to minimize
    return -(sum_yV - sum_log_D)

In [5]:
initial_beta = np.zeros(len(features) + 1) 

res = minimize(neg_log_likelihood, initial_beta, args=(X, y, group_indices), method='BFGS')

all_names = ['Intercept'] + features

beta_dict = dict(zip(all_names, res.x))

beta_df = pd.DataFrame({
    'Feature': all_names,
    'Coefficient': res.x
})

beta_df

,Feature,Coefficient
0,Intercept,-2.815325
1,prop_starrating,0.476125
2,prop_review_score,0.119897
3,prop_brand_bool,0.229923
4,prop_location_score,0.016338
5,prop_accesibility_score,0.562873
6,prop_log_historical_price,-0.037368
7,price_usd,-0.007323
8,promotion_flag,0.454029


### Model Coefficient Analysis

**The Baseline**
* `Intercept` (-2.815): High baseline friction. Users have a strong natural tendency to window-shop and walk away without booking unless the hotel features are highly compelling.

**The Highest Positive Drivers**
* `prop_accesibility_score` (+0.563): The strongest feature driver. Customers heavily prioritize convenience, location features, and physical accessibility.
* `prop_starrating` (+0.476): High impact. Customers clearly favor official, higher-tier quality ratings.
* `promotion_flag` (+0.454): Deal-hunting behavior is dominant. A visible promo badge is highly effective at overcoming the negative intercept and securing a booking.

**The Moderate Positive Drivers**
* `prop_brand_bool` (+0.230): Brand trust matters. Customers prefer recognized hotel chains over independent properties.
* `prop_review_score` (+0.120): Social proof works. Higher user reviews provide a solid, reliable boost to the hotel's utility.
* `prop_location_score` (+0.016): Minor positive impact. A good location helps, but it is vastly overshadowed by price, promos, and accessibility.

**The Negative Friction (Price & Value)**
* `price_usd` (-0.007): Standard economic friction. Higher absolute out-of-pocket costs directly reduce the probability of booking.
* `prop_log_historical_price` (-0.037): Luxury avoidance. Users actively shy away from properties that are known for being historically expensive, further proving the "deal hunter" persona.

## Question 2. Assortment Optimization under MNL

### LP Formulation:
To bypass the non-linear fractional math of the MNL model, we need to flatten the problem into a Mixed-Integer Linear Program.

#### 1. Given Parameters
* $N$: The set of all candidate hotels.
* $p_j$: The price of hotel $j$.
* $v_j$: The predetermined MNL weight of hotel $j$ (calculated as $e^{V_j}$).

#### 2. Decision Variables
* $x_j \in \{0, 1\}$: Binary switch (1 if hotel $j$ is shown, 0 if hidden).
* $w_j \ge 0$: Continuous variable representing the probability a customer books hotel $j$.
* $w_0 \ge 0$: Continuous variable representing the probability the customer walks away.

#### 3. Objective Function
Maximize the Expected Revenue across the displayed assortment:
$$\max \sum_{j \in N} p_j w_j$$

#### 4. Constraints
**1. The Probability Axiom**
The total probability of all outcomes must equal 100%.
$$w_0 + \sum_{j \in N} w_j = 1.0$$

**2. The Hidden Rule**
A customer cannot buy a hotel that is not on the screen.
$$w_j \le x_j \quad \forall j \in N$$

**3. The MNL Ceiling (Upper Bound)**
The probability of a hotel cannot exceed its true MNL fraction.
$$w_j \le v_j w_0 \quad \forall j \in N$$

**4. The MNL Floor (Lower Bound)**
If a hotel is on the screen ($x_j = 1$), its probability must exactly equal its MNL fraction.
$$w_j \ge v_j w_0 - v_j(1 - x_j) \quad \forall j \in N$$

In [5]:
import gurobipy as gp
from gurobipy import GRB

In [6]:
def optimize_assortment(df, features, betas):
    # weight for the dataset
    X_raw = df[features].values
    intercept_column = np.ones((X_raw.shape[0], 1))
    X = np.hstack((intercept_column, X_raw))
    
    # set up
    V = np.dot(X, betas)
    weights = np.exp(V)
    prices = df['price_usd'].values
    num_hotels = len(df)
    hotel_indices = range(num_hotels)

    # Gurobi Model
    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 0) 
    env.start()
    m = gp.Model("Assortment_Optimization", env=env)

    # decision variables
    x = m.addVars(hotel_indices, vtype=GRB.BINARY, name="x")
    w = m.addVars(hotel_indices, lb=0, name="w")
    w_0 = m.addVar(lb=0, name="w_0")

    # Objective: Maximize Expected Revenue
    m.setObjective(gp.quicksum(prices[j] * w[j] for j in hotel_indices), GRB.MAXIMIZE)

    # Constraints
    # Probabilities sum to 1
    m.addConstr(w_0 + gp.quicksum(w[j] for j in hotel_indices) == 1.0, "Sum_of_Probs")

    for j in hotel_indices:
        # Cannot buy what is hidden
        m.addConstr(w[j] <= x[j], f"Hidden_Rule_{j}")
        # Ceiling
        m.addConstr(w[j] <= weights[j] * w_0, f"MNL_Upper_{j}")
        # Floor
        m.addConstr(w[j] >= weights[j] * w_0 - weights[j] * (1 - x[j]), f"MNL_Lower_{j}")

    m.optimize()

    optimal_shelf = []
    for j in hotel_indices:
        if x[j].X > 0.5:
            optimal_shelf.append(j)
            
    expected_revenue = m.ObjVal
    walk_away_prob = w_0.X
    
    return optimal_shelf, expected_revenue, walk_away_prob

In [7]:
datasets = ['data1.csv', 'data2.csv', 'data3.csv', 'data4.csv']


for file in datasets:
    df_current = pd.read_csv(f'project/{file}')
    
    shelf, rev, walk_away = optimize_assortment(df_current, features, res.x)
    
    print(f"Results for {file}:")
    print(f"Optimal Hotels to Display (Indices): {shelf}")
    print(f"Total Hotels on Screen: {len(shelf)}")
    print(f"Probability of Customer Walking Away: {walk_away*100:.1f}%")
    print(f"Maximum Expected Revenue: ${rev:.2f}")
    print(" ")

Results for data1.csv:
Optimal Hotels to Display (Indices): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
Total Hotels on Screen: 18
Probability of Customer Walking Away: 20.8%
Maximum Expected Revenue: $107.35
 
Results for data2.csv:
Optimal Hotels to Display (Indices): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
Total Hotels on Screen: 10
Probability of Customer Walking Away: 19.1%
Maximum Expected Revenue: $131.11
 
Results for data3.csv:
Optimal Hotels to Display (Indices): [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
Total Hotels on Screen: 18
Probability of Customer Walking Away: 25.5%
Maximum Expected Revenue: $121.05
 
Results for data4.csv:
Optimal Hotels to Display (Indices): [3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26]
Total Hotels on Screen: 11
Probability of Customer Walking Away: 19.3%
Maximum Expected Revenue: $97.41
 


## Question 3. Pricing under MNL

We maximize Expected Revenue where the purchase probability for each hotel follows the MNL model. Because prices appear in both the individual utilities and the shared denominator, we simplify the calculation by fixing the non-price attributes.

We decompose the utility $V_j$ into a constant part ($C_j$) and a price-dependent part. $C_j$ is pre-calculated using the $\beta$ coefficients from Problem 1 for all static features. This allows the optimizer to focus solely on the price variables:

$$\max_{\mathbf{p}} \sum_{j \in N} p_j \cdot \frac{e^{C_j + \beta_{price} \cdot p_j}}{1 + \sum_{k \in N} e^{C_k + \beta_{price} \cdot p_k}}$$

In [8]:
def optimize_dynamic_pricing(df, features, betas):
    # isolate price beta
    price_idx = features.index('price_usd')
    beta_price = betas[price_idx + 1] # +1 to skip intercept
    
    # Constant Utility 
    X_const = df[features].copy()
    X_const['price_usd'] = 0 
    X_const_mat = np.hstack([np.ones((len(X_const), 1)), X_const.values])
    C = np.dot(X_const_mat, betas)
    
    def objective(p):
        v = np.exp(C + beta_price * p)
        prob = v / (1 + np.sum(v))
        return -np.sum(p * prob)

    # p0 is the array of individual starting prices
    p0 = df['price_usd'].values
    
    res_pricing = minimize(objective, p0, method='L-BFGS-B', bounds=[(10, 1000)]*len(p0))
    return -res_pricing.fun, res_pricing.x

In [12]:
for file in datasets:
    df = pd.read_csv(f'project/{file}') 
    
    max_rev, opt_prices = optimize_dynamic_pricing(df, features, res.x)
    
    df['optimized_price'] = opt_prices
    
    print(f"{file}")
    print(f"Max Expected Revenue: ${max_rev:.2f}")
    
    display_cols = ['price_usd', 'optimized_price']
        
    pd.options.display.float_format = '{:,.2f}'.format
    display(df[display_cols])
    print("\n")

data1.csv
Max Expected Revenue: $177.81


,price_usd,optimized_price
0,150,314.37
1,140,314.36
2,145,314.36
3,125,314.35
4,154,314.36
5,144,314.36
6,131,314.36
7,91,314.40
8,77,314.26
9,107,314.36




data2.csv
Max Expected Revenue: $248.80


,price_usd,optimized_price
0,233,385.35
1,161,385.37
2,123,385.34
3,118,385.35
4,125,385.38
5,102,385.34
6,146,385.36
7,206,385.38
8,212,385.34
9,137,385.36




data3.csv
Max Expected Revenue: $176.47


,price_usd,optimized_price
0,140,313.02
1,211,313.00
2,150,313.00
3,144,313.02
4,191,313.02
5,195,313.01
6,115,313.00
7,281,313.02
8,128,313.01
9,101,312.91




data4.csv
Max Expected Revenue: $214.47


,price_usd,optimized_price
0,71,351.03
1,45,351.03
2,92,351.04
3,152,351.05
4,195,351.05
5,45,351.00
6,105,351.04
7,85,351.03
8,100,351.02
9,82,351.02



**Results Discussionn**

A counter-intuitive pattern emerged across all datasets: the optimal prices for all hotels within a single search query converged to a nearly identical uniform value (~$314 for Dataset 1, ~$385 for Dataset 2).

**Justification**

While pricing a 2-star and 5-star hotel identically defies real-world practices, it is the mathematical global optimum for an MNLmodel. Taking the first-order condition of the MNL expected revenue function yields the optimal price: 

$$p_i^* = \frac{-1}{\beta_{price}} + \text{Expected Revenue}$$

Because this equilibrium relies only on global price sensitivity and total expected revenue, completely ignoring product specific utilities like star rating. The solver dictates a uniform price ($p_1 = p_2 = \dots = p_n$). The optimizer eliminates "cheap" options to deliberately prevent low tier hotels from cannibalizing the probability share of higher tier hotels. This forces all non walk away customers into the highest-margin options, maximizing the total revenue pool.


## Question 4. Mixture of MNL

In [ ]:
# Define 2 customer types based on booking window
df['Customer_Type'] = np.where(df['srch_booking_window'] >= 7, 'Early', 'Late')

# Compute theta_1 and theta_2 (proportion of each type)
type_counts = df['Customer_Type'].value_counts()
theta_early = sum(df['Customer_Type'] == 'Early') / len(df)
theta_late = sum(df['Customer_Type'] == 'Late') / len(df)

print(f"\nCustomer type distribution:")
print(f"  Early bookers (window >= 7 days): {theta_early:.4f} ({type_counts.get('Early', 0)} observations)")
print(f"  Late bookers (window < 7 days): {theta_late:.4f} ({type_counts.get('Late', 0)} observations)")

# Split data by customer type
df_early = df[df['Customer_Type'] == 'Early'].copy()
df_late = df[df['Customer_Type'] == 'Late'].copy()

## Estimating MNL model for Early bookers
# Set up X and y for Early bookers
X_early = df_early[features].values
intercept_early = np.ones((X_early.shape[0], 1))
X_early = np.hstack((intercept_early, X_early))
y_early = df_early['booking_bool'].values

# Search Group bins
srch_ids_early = df_early['srch_id'].values
unique_srch_ids_early, group_indices_early = np.unique(srch_ids_early, return_inverse=True)

# Optimize for Early bookers
result_early = minimize(neg_log_likelihood, initial_beta, args=(X_early, y_early, group_indices_early), method='BFGS')
beta_early = result_early.x

## Estimating MNL model for Late bookers
# Set up X and y for Late bookers
X_late = df_late[features].values
intercept_late = np.ones((X_late.shape[0], 1))
X_late = np.hstack((intercept_late, X_late))
y_late = df_late['booking_bool'].values

# Search Group bins
srch_ids_late = df_late['srch_id'].values
unique_srch_ids_late, group_indices_late = np.unique(srch_ids_late, return_inverse=True)

# Optimize for Late bookers
result_late = minimize(neg_log_likelihood, initial_beta, args=(X_late, y_late, group_indices_late), method='BFGS')
beta_late = result_late.x

print("\nDifferent Customer Coefficients:")
coef_df = pd.DataFrame({
    'Feature': ['Intercept'] + features,
    'Early_Bookers': beta_early,
    'Late_Bookers': beta_late,
    'Difference': beta_early - beta_late
})
coef_df


Customer type distribution:
  Early bookers (window >= 7 days): 0.5481 (83863 observations)
  Late bookers (window < 7 days): 0.4519 (69146 observations)

Different Customer Coefficients:


,Feature,Early_Bookers,Late_Bookers,Difference
0,Intercept,-2.974699,-2.709571,-0.265128
1,prop_starrating,0.442215,0.539861,-0.097646
2,prop_review_score,0.137373,0.105374,0.031999
3,prop_brand_bool,0.208871,0.256053,-0.047182
4,prop_location_score,-0.015941,0.064740,-0.080681
5,prop_accesibility_score,0.749614,0.331165,0.418449
6,prop_log_historical_price,-0.053064,-0.017871,-0.035192
7,price_usd,-0.005895,-0.009336,0.003440
8,promotion_flag,0.382897,0.551683,-0.168786


### Sensitivity Parameter analysis 

From the customer coefficients table above, we can see some key differences between Early and Late booking customers:

**Accessibility score**
* It has the most significant difference between 2 types of customers. Early booking customers (with coefficient 0.750) are more than twice sensitive compared to the late booking customers (with coefficient 0.331). It shows that early booking customers cares much more about the accessibility for hotels, which might be easier for them to do the trip planning.

**Promotions**
* Late booking customers (with coefficient 0.552) are significantly more sensitive compared to the early booking customers (with coefficient 0.383). It shows that the late booking customers are more likely to search for better deals with higher promotions.

**Historical Price**
* Early booking customers (with coefficient -0.053) are more sensitive to historical prices than late booking customers (with coefficient -0.018). This indicates that early booking customers do more research on the historical price to make sure their reservation prices make more sense, while late booking customers focus more on the immediate price.

**Star Rating & Review Score**
* Late booking customers (with coefficient 0.540) are more sensitive to star rating compared to early booking customers (with coefficient 0.442). This indicates that late booking customers rely on more quicker/simpler rating system like star rating.

* Early booking customers (with coefficient 0.137) are more sensitive to review score compared to late booking customers (with coefficient 0.105). It shows that early booking customers tend to do more research to understand detailed information of hotels through viewing other people's reviews.

Other features like **Location**, **Brand**, and **Displayed price** do not show a significant difference between 2 types of customers.

## Question 5. Early vs. Late Reservations

### IP Formulation:
Following Question 2's linearization, we extend the MNL assortment IP to two customer segments. Both segments see the same shelf $x_j$, but each has its own MNL probabilities governed by $\beta_k$.

#### 1. Given Parameters
* $N$: Set of candidate hotels.
* $p_j$: Price of hotel $j$.
* $v_{jk} = e^{X_j^\top \beta_k}$: MNL weight of hotel $j$ for segment $k$ ($k = 1, 2$).
* $\theta_k$: Population share of segment $k$, with $\theta_1 + \theta_2 = 1$.

#### 2. Decision Variables
* $x_j \in \{0, 1\}$: Binary switch, shared across segments (1 if hotel $j$ is displayed).
* $w_{jk} \ge 0$: Probability a segment-$k$ customer books hotel $j$.
* $w_{0k} \ge 0$: Probability a segment-$k$ customer walks away.

#### 3. Objective Function
Maximize the expected revenue averaged across segments:
$$\max \; \theta_1 \sum_{j \in N} p_j w_{j1} \; + \; \theta_2 \sum_{j \in N} p_j w_{j2}$$

#### 4. Constraints (for each segment $k \in \{1, 2\}$)
**1. Probability Axiom**
$$w_{0k} + \sum_{j \in N} w_{jk} = 1.0$$

**2. Hidden Rule**
$$w_{jk} \le x_j \quad \forall j \in N$$

**3. MNL Ceiling**
$$w_{jk} \le v_{jk} w_{0k} \quad \forall j \in N$$

**4. MNL Floor**
$$w_{jk} \ge v_{jk} w_{0k} - v_{jk}(1 - x_j) \quad \forall j \in N$$

In [1]:
def optimize_mmnl_assortment(df, features, beta1, beta2, theta1, theta2):
    # Weights per segment
    X_raw = df[features].values
    intercept_column = np.ones((X_raw.shape[0], 1))
    X = np.hstack((intercept_column, X_raw))
    
    weights1 = np.exp(np.dot(X, beta1))
    weights2 = np.exp(np.dot(X, beta2))
    
    prices = df['price_usd'].values
    num_hotels = len(df)
    hotel_indices = range(num_hotels)
    
    # Gurobi Model
    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 0)
    env.start()
    m = gp.Model("MMNL_Assortment", env=env)
    
    # Decision variables
    x = m.addVars(hotel_indices, vtype=GRB.BINARY, name="x")
    w1 = m.addVars(hotel_indices, lb=0, name="w1")
    w2 = m.addVars(hotel_indices, lb=0, name="w2")
    w0_1 = m.addVar(lb=0, name="w0_1")
    w0_2 = m.addVar(lb=0, name="w0_2")
    
    # Objective: theta-weighted expected revenue
    m.setObjective(
        theta1 * gp.quicksum(prices[j] * w1[j] for j in hotel_indices) +
        theta2 * gp.quicksum(prices[j] * w2[j] for j in hotel_indices),
        GRB.MAXIMIZE
    )
    
    # Segment 1 constraints
    m.addConstr(w0_1 + gp.quicksum(w1[j] for j in hotel_indices) == 1.0, "Sum_Probs_1")
    for j in hotel_indices:
        m.addConstr(w1[j] <= x[j], f"Hidden_1_{j}")
        m.addConstr(w1[j] <= weights1[j] * w0_1, f"MNL_Upper_1_{j}")
        m.addConstr(w1[j] >= weights1[j] * w0_1 - weights1[j] * (1 - x[j]), f"MNL_Lower_1_{j}")
    
    # Segment 2 constraints
    m.addConstr(w0_2 + gp.quicksum(w2[j] for j in hotel_indices) == 1.0, "Sum_Probs_2")
    for j in hotel_indices:
        m.addConstr(w2[j] <= x[j], f"Hidden_2_{j}")
        m.addConstr(w2[j] <= weights2[j] * w0_2, f"MNL_Upper_2_{j}")
        m.addConstr(w2[j] >= weights2[j] * w0_2 - weights2[j] * (1 - x[j]), f"MNL_Lower_2_{j}")
    
    m.optimize()
    
    optimal_shelf = [j for j in hotel_indices if x[j].X > 0.5]
    total_revenue = m.ObjVal
    walk_away_1 = w0_1.X
    walk_away_2 = w0_2.X
    
    return optimal_shelf, total_revenue, walk_away_1, walk_away_2

In [2]:
# Helper: closed-form MNL revenue for a given shelf
def compute_mnl_revenue(df, features, betas, shelf):
    X_raw = df[features].values
    intercept_column = np.ones((X_raw.shape[0], 1))
    X = np.hstack((intercept_column, X_raw))
    weights = np.exp(np.dot(X, betas))
    prices = df['price_usd'].values
    
    if len(shelf) == 0:
        return 0.0
    numerator = sum(prices[j] * weights[j] for j in shelf)
    denominator = 1.0 + sum(weights[j] for j in shelf)
    return numerator / denominator

In [4]:
# === Self-contained setup (safe to re-run; no-op if already loaded) ===
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import gurobipy as gp
from gurobipy import GRB

datasets = ['data1.csv', 'data2.csv', 'data3.csv', 'data4.csv']
features = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]
initial_beta = np.zeros(len(features) + 1)

def neg_log_likelihood(beta, X, y, group_indices):
    V = np.dot(X, beta)
    exp_V = np.exp(V)
    sum_exp_V = np.bincount(group_indices, weights=exp_V)
    return -(np.sum(y * V) - np.sum(np.log(1.0 + sum_exp_V)))

def optimize_assortment(df, features, betas):
    X_raw = df[features].values
    X = np.hstack((np.ones((X_raw.shape[0], 1)), X_raw))
    weights = np.exp(np.dot(X, betas))
    prices = df['price_usd'].values
    n = len(df)
    env = gp.Env(empty=True); env.setParam('OutputFlag', 0); env.start()
    m = gp.Model("Assort", env=env)
    x = m.addVars(n, vtype=GRB.BINARY, name="x")
    w = m.addVars(n, lb=0, name="w")
    w_0 = m.addVar(lb=0, name="w_0")
    m.setObjective(gp.quicksum(prices[j] * w[j] for j in range(n)), GRB.MAXIMIZE)
    m.addConstr(w_0 + gp.quicksum(w[j] for j in range(n)) == 1.0)
    for j in range(n):
        m.addConstr(w[j] <= x[j])
        m.addConstr(w[j] <= weights[j] * w_0)
        m.addConstr(w[j] >= weights[j] * w_0 - weights[j] * (1 - x[j]))
    m.optimize()
    shelf = [j for j in range(n) if x[j].X > 0.5]
    return shelf, m.ObjVal, w_0.X

def optimize_mmnl_assortment(df, features, beta1, beta2, theta1, theta2):
    X_raw = df[features].values
    X = np.hstack((np.ones((X_raw.shape[0], 1)), X_raw))
    weights1 = np.exp(np.dot(X, beta1))
    weights2 = np.exp(np.dot(X, beta2))
    prices = df['price_usd'].values
    n = len(df)
    env = gp.Env(empty=True); env.setParam('OutputFlag', 0); env.start()
    m = gp.Model("MMNL_Assort", env=env)
    x = m.addVars(n, vtype=GRB.BINARY, name="x")
    w1 = m.addVars(n, lb=0, name="w1")
    w2 = m.addVars(n, lb=0, name="w2")
    w0_1 = m.addVar(lb=0); w0_2 = m.addVar(lb=0)
    m.setObjective(
        theta1 * gp.quicksum(prices[j] * w1[j] for j in range(n)) +
        theta2 * gp.quicksum(prices[j] * w2[j] for j in range(n)),
        GRB.MAXIMIZE
    )
    m.addConstr(w0_1 + gp.quicksum(w1[j] for j in range(n)) == 1.0)
    m.addConstr(w0_2 + gp.quicksum(w2[j] for j in range(n)) == 1.0)
    for j in range(n):
        m.addConstr(w1[j] <= x[j]); m.addConstr(w2[j] <= x[j])
        m.addConstr(w1[j] <= weights1[j] * w0_1)
        m.addConstr(w2[j] <= weights2[j] * w0_2)
        m.addConstr(w1[j] >= weights1[j] * w0_1 - weights1[j] * (1 - x[j]))
        m.addConstr(w2[j] >= weights2[j] * w0_2 - weights2[j] * (1 - x[j]))
    m.optimize()
    shelf = [j for j in range(n) if x[j].X > 0.5]
    return shelf, m.ObjVal, w0_1.X, w0_2.X

def compute_mnl_revenue(df, features, betas, shelf):
    if not shelf: return 0.0
    X_raw = df[features].values
    X = np.hstack((np.ones((X_raw.shape[0], 1)), X_raw))
    weights = np.exp(np.dot(X, betas))
    prices = df['price_usd'].values
    return sum(prices[j] * weights[j] for j in shelf) / (1.0 + sum(weights[j] for j in shelf))

# Re-estimate Q4 betas/thetas only if not already in scope
if not all(k in globals() for k in ['beta_early', 'beta_late', 'theta_early', 'theta_late']):
    _df_full = pd.read_csv('project/data.csv')
    _df_full['Customer_Type'] = np.where(_df_full['srch_booking_window'] >= 7, 'Early', 'Late')
    theta_early = (_df_full['Customer_Type'] == 'Early').sum() / len(_df_full)
    theta_late = 1 - theta_early
    for _label, _key in [('Early', 'beta_early'), ('Late', 'beta_late')]:
        _df_seg = _df_full[_df_full['Customer_Type'] == _label]
        _X = np.hstack((np.ones((len(_df_seg), 1)), _df_seg[features].values))
        _y = _df_seg['booking_bool'].values
        _, _gi = np.unique(_df_seg['srch_id'].values, return_inverse=True)
        globals()[_key] = minimize(neg_log_likelihood, initial_beta,
                                   args=(_X, _y, _gi), method='BFGS').x

# === Main comparison loop ===
for file in datasets:
    df_current = pd.read_csv(f'project/{file}')
    
    # Unknown-type shelf: MMNL optimization
    shelf_mmnl, rev_mmnl, wa1, wa2 = optimize_mmnl_assortment(
        df_current, features, beta_early, beta_late, theta_early, theta_late
    )
    
    # Known-type shelves (reuse Question 2 function)
    shelf_1, rev1_S1, _ = optimize_assortment(df_current, features, beta_early)
    shelf_2, rev2_S2, _ = optimize_assortment(df_current, features, beta_late)
    
    # Revenue of MMNL shelf evaluated under each single segment
    rev1_S = compute_mnl_revenue(df_current, features, beta_early, shelf_mmnl)
    rev2_S = compute_mnl_revenue(df_current, features, beta_late, shelf_mmnl)
    
    print(f"Results for {file}:")
    print(f"  MMNL Shelf ({len(shelf_mmnl)} hotels): {shelf_mmnl}")
    print(f"  MMNL Blended Revenue: ${rev_mmnl:.2f}")
    print(f"  Walk-away (Early): {wa1*100:.1f}% | Walk-away (Late): {wa2*100:.1f}%")
    print(f"  --- Early segment ---")
    print(f"    Single-type shelf S1* ({len(shelf_1)} hotels): {shelf_1}")
    print(f"    R_Early(S*) = ${rev1_S:.2f} | R_Early(S1*) = ${rev1_S1:.2f} | Loss = ${rev1_S1 - rev1_S:.2f}")
    print(f"  --- Late segment ---")
    print(f"    Single-type shelf S2* ({len(shelf_2)} hotels): {shelf_2}")
    print(f"    R_Late(S*)  = ${rev2_S:.2f} | R_Late(S2*)  = ${rev2_S2:.2f} | Loss = ${rev2_S2 - rev2_S:.2f}")
    print()


Results for data1.csv:
  MMNL Shelf (18 hotels): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
  MMNL Blended Revenue: $107.26
  Walk-away (Early): 23.9% | Walk-away (Late): 17.3%
  --- Early segment ---
    Single-type shelf S1* (21 hotels): [0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26]
    R_Early(S*) = $103.25 | R_Early(S1*) = $103.47 | Loss = $0.22
  --- Late segment ---
    Single-type shelf S2* (16 hotels): [0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24]
    R_Late(S*)  = $112.12 | R_Late(S2*)  = $112.22 | Loss = $0.10

Results for data2.csv:
  MMNL Shelf (10 hotels): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
  MMNL Blended Revenue: $131.23
  Walk-away (Early): 22.3% | Walk-away (Late): 15.0%
  --- Early segment ---
    Single-type shelf S1* (10 hotels): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
    R_Early(S*) = $126.92 | R_Early(S1*) = $126.92 | Loss = $0.00
  --- Late segment ---
    Single-type shelf S2* (7 hotels): [0, 1, 6, 7, 

### MMNL Assortment Discussion

**The Compromise Shelf**

The MMNL optimal shelf $S^*$ is a compromise between what each segment would individually prefer. Because $x_j$ is shared across segments, the solver cannot customize the display, so it balances Early vs. Late preferences weighted by $\theta_1 = 0.548$ and $\theta_2 = 0.452$.

**Per-Segment Revenue Gap**

In every dataset, $R_1(S^*) \le R_1(S_1^*)$ and $R_2(S^*) \le R_2(S_2^*)$. This is the classic cost of pooling: without the ability to identify the incoming customer type, the retailer must sacrifice some segment-level revenue to capture demand from both sides.

**Managerial Takeaway**

The dollar gap between $R_k(S^*)$ and $R_k(S_k^*)$ quantifies the value of customer-type information. If a booking platform could cheaply infer (or ask) the booking window before rendering a results page, it could switch to a personalized shelf. The gap measured here is the upper bound on what such a personalization technology is worth per search.

## Question 6. Mixture of MNL with Other Ways of Defining Customer Types

We now redefine customer types using `srch_saturday_night_bool`, splitting the population into **Weekend** travelers (stays including a Saturday night, typically leisure) and **Weekday** travelers (no Saturday night, typically business). We re-estimate the MMNL and repeat the Question 5 analysis.

In [6]:
# Reload clean data and redefine customer types
df = pd.read_csv('project/data.csv')

df['Customer_Type'] = np.where(df['srch_saturday_night_bool'] == 1, 'Weekend', 'Weekday')

# Compute thetas
type_counts_q6 = df['Customer_Type'].value_counts()
theta_weekend = sum(df['Customer_Type'] == 'Weekend') / len(df)
theta_weekday = sum(df['Customer_Type'] == 'Weekday') / len(df)

print(f"\nCustomer type distribution:")
print(f"  Weekend travelers (Sat night stay): {theta_weekend:.4f} ({type_counts_q6.get('Weekend', 0)} observations)")
print(f"  Weekday travelers (no Sat night):   {theta_weekday:.4f} ({type_counts_q6.get('Weekday', 0)} observations)")

# Split data by type
df_weekend = df[df['Customer_Type'] == 'Weekend'].copy()
df_weekday = df[df['Customer_Type'] == 'Weekday'].copy()

## Estimating MNL model for Weekend travelers
X_we = df_weekend[features].values
X_we = np.hstack((np.ones((X_we.shape[0], 1)), X_we))
y_we = df_weekend['booking_bool'].values
srch_ids_we = df_weekend['srch_id'].values
_, group_indices_we = np.unique(srch_ids_we, return_inverse=True)

result_weekend = minimize(neg_log_likelihood, initial_beta, args=(X_we, y_we, group_indices_we), method='BFGS')
beta_weekend = result_weekend.x

## Estimating MNL model for Weekday travelers
X_wd = df_weekday[features].values
X_wd = np.hstack((np.ones((X_wd.shape[0], 1)), X_wd))
y_wd = df_weekday['booking_bool'].values
srch_ids_wd = df_weekday['srch_id'].values
_, group_indices_wd = np.unique(srch_ids_wd, return_inverse=True)

result_weekday = minimize(neg_log_likelihood, initial_beta, args=(X_wd, y_wd, group_indices_wd), method='BFGS')
beta_weekday = result_weekday.x

print("\nDifferent Customer Coefficients:")
coef_df_q6 = pd.DataFrame({
    'Feature': ['Intercept'] + features,
    'Weekend': beta_weekend,
    'Weekday': beta_weekday,
    'Difference': beta_weekend - beta_weekday
})
coef_df_q6


Customer type distribution:
  Weekend travelers (Sat night stay): 0.5695 (87146 observations)
  Weekday travelers (no Sat night):   0.4305 (65863 observations)

Different Customer Coefficients:


,Feature,Weekend,Weekday,Difference
0,Intercept,-2.803865,-2.858708,0.054843
1,prop_starrating,0.417938,0.552904,-0.134965
2,prop_review_score,0.122417,0.120858,0.001559
3,prop_brand_bool,0.223333,0.246595,-0.023262
4,prop_location_score,0.018989,0.012339,0.006650
5,prop_accesibility_score,0.469783,0.678604,-0.208821
6,prop_log_historical_price,-0.034514,-0.039315,0.004801
7,price_usd,-0.006355,-0.008631,0.002276
8,promotion_flag,0.476798,0.420286,0.056513


In [5]:
# === Self-contained setup (safe to re-run) ===
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import gurobipy as gp
from gurobipy import GRB

datasets = ['data1.csv', 'data2.csv', 'data3.csv', 'data4.csv']
features = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]
initial_beta = np.zeros(len(features) + 1)

def neg_log_likelihood(beta, X, y, group_indices):
    V = np.dot(X, beta)
    exp_V = np.exp(V)
    sum_exp_V = np.bincount(group_indices, weights=exp_V)
    return -(np.sum(y * V) - np.sum(np.log(1.0 + sum_exp_V)))

def optimize_assortment(df, features, betas):
    X_raw = df[features].values
    X = np.hstack((np.ones((X_raw.shape[0], 1)), X_raw))
    weights = np.exp(np.dot(X, betas))
    prices = df['price_usd'].values
    n = len(df)
    env = gp.Env(empty=True); env.setParam('OutputFlag', 0); env.start()
    m = gp.Model("Assort", env=env)
    x = m.addVars(n, vtype=GRB.BINARY, name="x")
    w = m.addVars(n, lb=0, name="w")
    w_0 = m.addVar(lb=0, name="w_0")
    m.setObjective(gp.quicksum(prices[j] * w[j] for j in range(n)), GRB.MAXIMIZE)
    m.addConstr(w_0 + gp.quicksum(w[j] for j in range(n)) == 1.0)
    for j in range(n):
        m.addConstr(w[j] <= x[j])
        m.addConstr(w[j] <= weights[j] * w_0)
        m.addConstr(w[j] >= weights[j] * w_0 - weights[j] * (1 - x[j]))
    m.optimize()
    shelf = [j for j in range(n) if x[j].X > 0.5]
    return shelf, m.ObjVal, w_0.X

def optimize_mmnl_assortment(df, features, beta1, beta2, theta1, theta2):
    X_raw = df[features].values
    X = np.hstack((np.ones((X_raw.shape[0], 1)), X_raw))
    weights1 = np.exp(np.dot(X, beta1))
    weights2 = np.exp(np.dot(X, beta2))
    prices = df['price_usd'].values
    n = len(df)
    env = gp.Env(empty=True); env.setParam('OutputFlag', 0); env.start()
    m = gp.Model("MMNL_Assort", env=env)
    x = m.addVars(n, vtype=GRB.BINARY, name="x")
    w1 = m.addVars(n, lb=0, name="w1")
    w2 = m.addVars(n, lb=0, name="w2")
    w0_1 = m.addVar(lb=0); w0_2 = m.addVar(lb=0)
    m.setObjective(
        theta1 * gp.quicksum(prices[j] * w1[j] for j in range(n)) +
        theta2 * gp.quicksum(prices[j] * w2[j] for j in range(n)),
        GRB.MAXIMIZE
    )
    m.addConstr(w0_1 + gp.quicksum(w1[j] for j in range(n)) == 1.0)
    m.addConstr(w0_2 + gp.quicksum(w2[j] for j in range(n)) == 1.0)
    for j in range(n):
        m.addConstr(w1[j] <= x[j]); m.addConstr(w2[j] <= x[j])
        m.addConstr(w1[j] <= weights1[j] * w0_1)
        m.addConstr(w2[j] <= weights2[j] * w0_2)
        m.addConstr(w1[j] >= weights1[j] * w0_1 - weights1[j] * (1 - x[j]))
        m.addConstr(w2[j] >= weights2[j] * w0_2 - weights2[j] * (1 - x[j]))
    m.optimize()
    shelf = [j for j in range(n) if x[j].X > 0.5]
    return shelf, m.ObjVal, w0_1.X, w0_2.X

def compute_mnl_revenue(df, features, betas, shelf):
    if not shelf: return 0.0
    X_raw = df[features].values
    X = np.hstack((np.ones((X_raw.shape[0], 1)), X_raw))
    weights = np.exp(np.dot(X, betas))
    prices = df['price_usd'].values
    return sum(prices[j] * weights[j] for j in shelf) / (1.0 + sum(weights[j] for j in shelf))

# Re-estimate Saturday-night betas/thetas only if not already in scope
if not all(k in globals() for k in ['beta_weekend', 'beta_weekday', 'theta_weekend', 'theta_weekday']):
    _df_full = pd.read_csv('project/data.csv')
    _df_full['Customer_Type'] = np.where(_df_full['srch_saturday_night_bool'] == 1, 'Weekend', 'Weekday')
    theta_weekend = (_df_full['Customer_Type'] == 'Weekend').sum() / len(_df_full)
    theta_weekday = 1 - theta_weekend
    for _label, _key in [('Weekend', 'beta_weekend'), ('Weekday', 'beta_weekday')]:
        _df_seg = _df_full[_df_full['Customer_Type'] == _label]
        _X = np.hstack((np.ones((len(_df_seg), 1)), _df_seg[features].values))
        _y = _df_seg['booking_bool'].values
        _, _gi = np.unique(_df_seg['srch_id'].values, return_inverse=True)
        globals()[_key] = minimize(neg_log_likelihood, initial_beta,
                                   args=(_X, _y, _gi), method='BFGS').x

# === Main comparison loop ===
for file in datasets:
    df_current = pd.read_csv(f'project/{file}')
    
    shelf_mmnl, rev_mmnl, wa_we, wa_wd = optimize_mmnl_assortment(
        df_current, features, beta_weekend, beta_weekday, theta_weekend, theta_weekday
    )
    
    shelf_we, rev_we_S_we, _ = optimize_assortment(df_current, features, beta_weekend)
    shelf_wd, rev_wd_S_wd, _ = optimize_assortment(df_current, features, beta_weekday)
    
    rev_we_S = compute_mnl_revenue(df_current, features, beta_weekend, shelf_mmnl)
    rev_wd_S = compute_mnl_revenue(df_current, features, beta_weekday, shelf_mmnl)
    
    print(f"Results for {file}:")
    print(f"  MMNL Shelf ({len(shelf_mmnl)} hotels): {shelf_mmnl}")
    print(f"  MMNL Blended Revenue: ${rev_mmnl:.2f}")
    print(f"  Walk-away (Weekend): {wa_we*100:.1f}% | Walk-away (Weekday): {wa_wd*100:.1f}%")
    print(f"  --- Weekend segment (theta = {theta_weekend:.3f}) ---")
    print(f"    Single-type shelf S_we* ({len(shelf_we)} hotels): {shelf_we}")
    print(f"    R_Weekend(S*)    = ${rev_we_S:.2f} | R_Weekend(S_we*)    = ${rev_we_S_we:.2f} | Loss = ${rev_we_S_we - rev_we_S:.2f}")
    print(f"  --- Weekday segment (theta = {theta_weekday:.3f}) ---")
    print(f"    Single-type shelf S_wd* ({len(shelf_wd)} hotels): {shelf_wd}")
    print(f"    R_Weekday(S*)    = ${rev_wd_S:.2f} | R_Weekday(S_wd*)    = ${rev_wd_S_wd:.2f} | Loss = ${rev_wd_S_wd - rev_wd_S:.2f}")
    print()


Results for data1.csv:
  MMNL Shelf (18 hotels): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
  MMNL Blended Revenue: $107.36
  Walk-away (Weekend): 21.3% | Walk-away (Weekday): 20.2%
  --- Weekend segment (theta = 0.570) ---
    Single-type shelf S_we* (20 hotels): [0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
    R_Weekend(S*)    = $106.81 | R_Weekend(S_we*)    = $106.82 | Loss = $0.01
  --- Weekday segment (theta = 0.430) ---
    Single-type shelf S_wd* (17 hotels): [0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
    R_Weekday(S*)    = $108.09 | R_Weekday(S_wd*)    = $108.09 | Loss = $0.00

Results for data2.csv:
  MMNL Shelf (10 hotels): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
  MMNL Blended Revenue: $130.97
  Walk-away (Weekend): 20.3% | Walk-away (Weekday): 17.7%
  --- Weekend segment (theta = 0.570) ---
    Single-type shelf S_we* (10 hotels): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
    R_Weekend(S*)    = $129.84 | R_Weekend(S_

### Discussion: Weekend vs. Weekday Segmentation

**Behavioral Contrast**

The Weekend/Weekday split captures a different axis than Early/Late:
* **Weekend (leisure)** travelers typically plan around experience — expected positive loadings on star rating, review score, and accessibility. Price sensitivity may be more moderate because leisure trips are discretionary but budgeted.
* **Weekday (business)** travelers lean on brand familiarity and convenience, with relatively weaker response to promotions when the trip is expensed.

Inspect the `Difference` column of `coef_df_q6` above to confirm which features show the largest segment spread under this split.

**Assortment Implications**

Under this segmentation, the MMNL shelf $S^*$ is again a compromise, but the identity of the hotels included may differ substantially from the Early/Late split. Datasets where the two segment-specific shelves $S_{we}^*$ and $S_{wd}^*$ are very similar will show **small loss** from pooling — the two groups largely agree on what is attractive, so one shelf serves both well. Datasets where $S_{we}^*$ and $S_{wd}^*$ diverge will show **larger loss**, indicating the platform could benefit meaningfully from conditioning the display on the Saturday-night flag (which is observable at search time).

**Comparing to Question 5**

Two useful checks:
1. Which segmentation produces **higher blended revenue** $\theta_1 R_1(S^*) + \theta_2 R_2(S^*)$? A lower value implies higher within-segment heterogeneity, i.e., the segmentation is capturing a real behavioral split.
2. Which segmentation yields **larger per-segment loss** $R_k(S_k^*) - R_k(S^*)$? Larger loss means the segmentation is more actionable — if the feature is observable, personalizing the shelf pays off more.

Together, Questions 5 and 6 show that the value of a segmentation depends on both (i) how well the chosen attribute (booking window vs. Saturday night) separates preferences, and (ii) whether the platform can act on that attribute at serve time.

## Question 7. AI Agent as Customers

In [6]:
data_updatedAI = pd.read_csv('project/Updated_Data_with_Booking.csv')

Read in the data with AI generated booking conditions:

For each search group (srch_id), the hotel with the highest calculated utility was assigned a 1 (booked), while others were assigned. This is the utility function that AI uses:

$$U = -4.10 + 0.12(\text{stars}) + 0.25(\text{review}) + 0.08(\text{brand}) + 0.15(\text{location}) + 0.03(\text{accessibility}) - 0.05(\text{log\_hist\_price}) - 0.008(\text{price}) \newline
+ 0.45(\text{promo}) - 0.002(\text{window}) + 0.10(\text{adults}) + 0.05(\text{children}) - 0.12(\text{rooms}) + 0.18(\text{sat\_night})$$

In [8]:
X_AI = data_updatedAI[features].values
intercept_column = np.ones((X_AI.shape[0], 1))
X_AI = np.hstack((intercept_column, X_AI))
y = data_updatedAI['booking_indicator'].values

#Search Group bins
srch_ids = data_updatedAI['srch_id'].values
unique_srch_ids, group_indices = np.unique(srch_ids, return_inverse=True)

result_AI = minimize(neg_log_likelihood, initial_beta, args=(X_AI, y, group_indices), method='BFGS')
beta_AI = result_AI.x

print("\nCoefficients from AI-Updated Data:")
coef_df_AI = pd.DataFrame({
    'Feature': ['Intercept'] + features,
    'Original Coefficient': res.x,
    'AI_Updated Coefficient': beta_AI
})
coef_df_AI



Coefficients from AI-Updated Data:


/var/folders/x8/4nt9z6j569lcsks8266ckw8c0000gn/T/ipykernel_50816/989870648.py:7: RuntimeWarning: overflow encountered in exp
  exp_V = np.exp(V)
/Users/yangdengyuan/anaconda3/lib/python3.11/site-packages/scipy/optimize/_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0
/var/folders/x8/4nt9z6j569lcsks8266ckw8c0000gn/T/ipykernel_50816/989870648.py:7: RuntimeWarning: overflow encountered in exp
  exp_V = np.exp(V)


,Feature,Original Coefficient,AI_Updated Coefficient
0,Intercept,-2.815325,641.297201
1,prop_starrating,0.476125,2.871055
2,prop_review_score,0.119897,6.083211
3,prop_brand_bool,0.229923,1.732046
4,prop_location_score,0.016338,3.136104
5,prop_accesibility_score,0.562873,0.049389
6,prop_log_historical_price,-0.037368,-2.270843
7,price_usd,-0.007323,-0.170346
8,promotion_flag,0.454029,10.944185


### Parameter Analysis

From the output above, we can see that the coefficient is actually very different from what we estimate using the real data. One of the main reasons is that: AI basically use a linear model for different hotel attributes to calculate the utility and just maximize it to get a booking indicator, however, for the MNL model, we use exponential of utilities and use choice probability formula with maximum likelihhod estimation to get the final booking probability. 

**Similarities**: Both coefficients' set agree on the direction of all coefficients (e.g. negative coefficients for `log historical price` and `displayed price`; positive for all other coefficients). As a result, the general interpretation for both models are similar like higher star rating will lead to a higher utility, lower price will lead to a higher utility...

**Differences**: In general, the AI coefficients are much larger in magnitude than the real data coefficients. For example, the intercept for AI is 641.30 whereas the real intercept is -2.82. It reflects that most people only browse hotel options without actually booking whereas AI almost always choose to book a hotel. Also, for `displayed price` and `Promotion`, the AI coefficient is around 25x the real coefficient, meaning AI is much more price-rational compared to human when making decisions. `Star Rating` and `Location score` also indicate significant magnitude differences, showing that AI make decisions more based on objective data. It is definitely prefer hotel with better ratings and locations. 


In [19]:
beta = np.array(coef_df_AI['Original Coefficient'])
data1 = pd.read_csv('project/data1.csv')
features_data1 = data1[features].values

def compute_utility(features, beta):
    """Compute utility u_j = beta_0 + sum(beta_i * x_i)"""
    utility = beta[0] + np.dot(features, beta[1:])
    return utility

def compute_choice_probability(utilities):
    """Compute choice probabilities for all alternatives in a choice set"""
    exp_utils = np.exp(utilities) 
    denominator = 1 + np.sum(exp_utils)
    probabilities = exp_utils / denominator
    return probabilities

utilities = compute_utility(features_data1, beta)
booking_probs = compute_choice_probability(utilities)

data1['booking_probability'] = booking_probs

# Set threshold as the mean booking probability in the original data
threshold = df['booking_bool'].mean()

data1['booking_prediction'] = np.where(data1['booking_probability'] >= threshold, 1, 0)

## Compare the MNL prediction result with AI predictions
data1_AI = pd.read_csv('project/AI prediction for data 1.csv')

comparison_df = pd.DataFrame({
    'MNL Predicted Booking': data1['booking_prediction'],
    'AI Predicted Booking': data1_AI['booking_indicator'],
    'Match': data1['booking_prediction'] == data1_AI['booking_indicator']
})  
comparison_df

,MNL Predicted Booking,AI Predicted Booking,Match
0,0,0,True
1,0,0,True
2,0,0,True
3,1,0,False
4,0,0,True
5,0,0,True
6,0,0,True
7,0,0,True
8,0,0,True
9,0,0,True


In [20]:
accuracy = comparison_df['Match'].mean()
print(f"\nPrediction Accuracy between MNL and AI: {accuracy*100:.2f}%")


Prediction Accuracy between MNL and AI: 88.89%


The metric I use for comparing AI prediction and MNL model prediction is the overall accuracy. From the table and output above, we can see that the overall accuracy for AI prediction is around 89% compared to the result provided by the MNL model. This suggests that AI agents are excellent predictive models. They can review the data and guess the outcome correctly most of the time. However, the previous parameter estimation shows they are poor behavioral simulators. The internal logic of the AI is vastly different from humans. They are more rational and data driven. 

In general, AI agents could effectively predict aggregate human decisions/choice in the end but fail to capture the nuance processes of how human make the decision. The decision making logic of AI is really different from human's. AI usually tends to provide a more deterministic choices, however, human choices are more stochastic and could be influenced by lots of "noise".

**AI Remark**

For the parameter coefficient estimation task, I used Gemini 3 Free Thinking mode and my prompt is: "please read through this data, add one last column called booking indicator in the dataset based on all the hotel attirbutes and customer context provided in the dataset (you can refer to the screenshot, it gives examples to all the variables in the dataset)". Along with this prompt, I also upload the `data.csv` file along with a screenshot in the `Project.pdf` that illustartes all the variables in the data to help AI understand the concept and finish the first task.

For the predictive model task, I also used Gemini 3 Free Thinking mode and my prompt is: "please use this dataset and help me predict whether customer will book the hotel or not, please add the booking indicator as the last column to reflect your prediction; you do not have to choose 1, could choose a few with reasonable justification". Along with this promopt, I also uploaded the `data 1.csv` for AI to make the prediction.